# Model comparison under attack

Train logistic regression, random forest and a small MLP on the same SWIFT dataset and compare clean vs robust accuracy and ASR at a fixed epsilon.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

import pandas as pd
from examples.adversarial import swift_binary_dataset, train_test, train, evaluate_robustness

import os
X, y, feats, target = swift_binary_dataset(2000, seed=int(os.environ.get('FOX_RUN_SEED', '42')))
Xtr, Xte, ytr, yte = train_test(X, y)
EPS = 0.5
rows = []
models = {}
for name in ['lr', 'rf', 'mlp']:
    m = train(name, Xtr, ytr)
    models[name] = m
    r = evaluate_robustness(m, Xte, yte, EPS)
    rows.append({'model': name.upper(), 'clean_acc': r['clean_accuracy'],
                 'robust_acc': r['robust_accuracy'], 'asr': r['attack_success_rate_on_correct']})
df = pd.DataFrame(rows)
print(df.to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt
x = range(len(df))
fig, ax = plt.subplots(figsize=(7.5, 4))
ax.bar([i - 0.2 for i in x], df['clean_acc'], width=0.4, label='clean accuracy', color='#4f8cff')
ax.bar([i + 0.2 for i in x], df['robust_acc'], width=0.4, label=f'robust accuracy (eps={EPS})', color='#e05b5b')
ax.set_xticks(list(x)); ax.set_xticklabels(df['model'])
ax.set_ylabel('accuracy'); ax.legend()
ax.set_title('Robustness by model family (SWIFT URGENCY task)')
for i, r in enumerate(df['asr']):
    ax.text(i + 0.2, 0.02, 'ASR %.0f%%' % (r * 100), fontsize=8, ha='center')
ax.grid(alpha=0.3, axis='y')